In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
hours = 24 * 365 # in a year

day_of_year = np.arange(hours) / 24
# not 1 2 3 .. to 365

seasonal =  15 + 25 * np.sin(2 * np.pi * (day_of_year - 150) / 365)
# phase 0 at day 150, hydrograph begins to explode from this
# macro cycle throughout the year
seasonal = np.clip(seasonal, 5, 60)

hour_of_day = np.arange(hours) % 24
diurnal = 1.0 + 0.15 * np.sin(2 * np.pi * (hour_of_day - 14) / 24)
# micro cycle throughout the day
# incr from hour 14 as upward trj
# amplitude reps ablation

# like time series, modeeling the seasonality blocks
# a0 + asint

noise = np.random.lognormal(mean=0, sigma=0.2, size=hours)
# modeling noise in multiplicative env

inflow = seasonal * diurnal * noise
inflow = np.clip(inflow, 2.0, 100.0)  # m³/s,

df = pd.DataFrame({
    "hour": range(hours),
    "inflow_m3s": inflow.round(2),
    "day_of_year": (np.arange(hours) / 24).astype(int),
    "hour_of_day": np.arange(hours) % 24,
})
df.to_csv("../data/synthetic_inflow_1yr.csv", index=False)

print(f"Generated {hours} hours ({hours/24:.0f} days) of inflow data")
print(f"Min: {inflow.min():.1f} m³/s, Max: {inflow.max():.1f} m³/s")
print(f"Mean: {inflow.mean():.1f} m³/s, Std: {inflow.std():.1f} m³/s")


In [ ]:
import matplotlib.pyplot as plt

#  Averages and zoomed-out blobs hide the physical risk.

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# (The Whole Year)
ax1.plot(df["hour"], df["inflow_m3s"], alpha=0.8, color="#1f77b4", linewidth=0.5)
ax1.set_title("1-Year Macroscopic View: The Seasonal Clip and Monsoon Peak", fontweight="bold")
ax1.set_xlabel("Hour of Year")
ax1.set_ylabel("Inflow (m³/s)")
ax1.set_xlim(0, hours)
ax1.grid(True, alpha=0.3)

#  Microscopic View (A 14-Day Monsoon window: e.g. Day 200 = hour 4800)
zoom_start = 4800
zoom_end = 4800 + (14 * 24)
ax2.plot(df["hour"][zoom_start:zoom_end], df["inflow_m3s"][zoom_start:zoom_end],
         color="#d62728", marker=".", markersize=4, linewidth=1)
ax2.set_title("14-Day Microscopic Truth (Day 200): Diurnal Cycling and Noise", fontweight="bold")
ax2.set_xlabel("Hour of Year")
ax2.set_ylabel("Inflow (m³/s)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()